In [203]:
import pandas as pd 
from pathlib import Path
import os

In [204]:
ROOT = Path(os.getcwd()).parents[1]
INPUT_PATH = Path(ROOT / "Data" / "Raw" / "catalogo_madrid_centro.csv")
print(INPUT_PATH)

/Users/santiago/Desktop/DSIA/groceries/Data/Raw/catalogo_madrid_centro.csv


In [205]:
df = pd.read_csv(INPUT_PATH)
df.head()

,cp,categoria,subcategoria,nombre,formato_y_precio_por_kilo,precio_unidad
0,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado","Garrafa 5 L | 3,35 €/L","16,75 €"
1,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado","Botella 1 L | 3,60 €/L","3,60 €"
2,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,"Garrafa 3 L | 3,75 €/L","11,25 €"
3,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,"Botella 1 L | 4,10 €/L","4,10 €"
4,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado Gran Se...,"Botella 750 ml | 7,00 €/L","5,25 €"


## Precio total

In [206]:
df["precio"] = df["precio_unidad"].str.replace(',','.').str.strip().str.removesuffix('€')
df = df.drop("precio_unidad", axis=1)
df.head()

,cp,categoria,subcategoria,nombre,formato_y_precio_por_kilo,precio
0,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado","Garrafa 5 L | 3,35 €/L",16.75
1,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado","Botella 1 L | 3,60 €/L",3.60
2,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,"Garrafa 3 L | 3,75 €/L",11.25
3,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,"Botella 1 L | 4,10 €/L",4.10
4,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado Gran Se...,"Botella 750 ml | 7,00 €/L",5.25


## Precio por kg, L, ud., ...

In [207]:
df["formato"] = df["formato_y_precio_por_kilo"].str.split('|').str[0].str.strip()
df["precio_por_"] = df["formato_y_precio_por_kilo"].str.split('|').str[1].str.strip()
df = df.drop("formato_y_precio_por_kilo", axis=1)
df.head()

,cp,categoria,subcategoria,nombre,precio,formato,precio_por_
0,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",16.75,Garrafa 5 L,"3,35 €/L"
1,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",3.60,Botella 1 L,"3,60 €/L"
2,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,11.25,Garrafa 3 L,"3,75 €/L"
3,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,4.10,Botella 1 L,"4,10 €/L"
4,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado Gran Se...,5.25,Botella 750 ml,"7,00 €/L"


In [208]:
df["precio_por_"].unique()

array(['3,35 €/L', '3,60 €/L', '3,75 €/L', ..., '0,932 €/100 ml',
       '13,00 €/ud', '8,50 €/ud'], shape=(1700,), dtype=object)

In [209]:
df["unidades"] = df["precio_por_"].str.split('/').str[-1]
df["unidades"].unique()

array(['L', 'kg', '100 g', 'ud', '100 ml', 'dz', 'dc', 'lv', 'm'],
      dtype=object)

### `dz`, `dc` -> docena

In [210]:
df.loc[df["unidades"] == 'dz', "unidades"] = 'docena'
df.loc[df["unidades"] == 'dc', "unidades"] = 'docena'

### `lv` -> `lavado`

In [211]:
df.loc[df["unidades"] == "lv", "unidades"] = 'lavado'

In [212]:
df["unidades"].unique()

array(['L', 'kg', '100 g', 'ud', '100 ml', 'docena', 'lavado', 'm'],
      dtype=object)

In [213]:
df["precio_por_"] = df["precio_por_"].str.split('€').str[0].str.strip().str.replace(',','.')

In [214]:
df.head()

,cp,categoria,subcategoria,nombre,precio,formato,precio_por_,unidades
0,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",16.75,Garrafa 5 L,3.35,L
1,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",3.60,Botella 1 L,3.60,L
2,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,11.25,Garrafa 3 L,3.75,L
3,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,4.10,Botella 1 L,4.10,L
4,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado Gran Se...,5.25,Botella 750 ml,7.00,L


## Formato

Dejamos las columnas del formato para el final

In [197]:
columna_formato = df["formato"]
df = df.drop("formato", axis=1)
df = pd.concat([df, columna_formato], axis=1)
df.head()

,cp,categoria,subcategoria,nombre,precio,precio_por_,unidades,formato
0,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",16.75,3.35,L,Garrafa 5 L
1,28012,"Aceite, especias y salsas","Aceite, vinagre y sal","Aceite de oliva 0,4º Hacendado",3.60,3.60,L,Botella 1 L
2,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,11.25,3.75,L,Garrafa 3 L
3,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado,4.10,4.10,L,Botella 1 L
4,28012,"Aceite, especias y salsas","Aceite, vinagre y sal",Aceite de oliva virgen extra Hacendado Gran Se...,5.25,7.00,L,Botella 750 ml


In [202]:
df["formato"].unique()

array(['Garrafa 5 L', 'Botella 1 L', 'Garrafa 3 L', ...,
       'Manojo 660 g aprox.', '1/2 Pieza 680 g aprox.',
       'Paquete 1,24 kg aprox.'], shape=(1318,), dtype=object)